# regrets-to-inform: A Data Science Skills Showcase
## Hypothesis: 
The South African Jobs market is broken for graduates. Jobs are mostly unavailable, hiring managers ghost applicants, and many jobs were never available at the time you clicked apply. Companies leave listings open for roles that are already filled, or in some extreme cases, appear to be engaging in data harvesting for ATS training. 

In [2]:
import numpy as np
from linkedin_parser import parse_jobs
import pandas as pd
import polars as pl
import seaborn as sns 
import json
import glob
from rapidfuzz import fuzz, utils
import uuid
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 7)
pd.set_option('display.precision', 8)
import re
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import ibis
import pyarrow
con = ibis.polars.connect()
ibis.options.interactive = True
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
FIG_W = 14

Let us set up the file processing pipeline

In [6]:
file_path1 = 'nonfunctionapps/unprocessed_email_dump/unprocessed'
file_path2 = 'nonfunctionapps/email_dump/processed'
EMPTY_LIST = ['Unknown', 'None', None, 'nan', 'none', 'NA', 'NaN', ""]

file_list = glob.glob(f'{file_path1}/*.json')
obj_list = []
for files in file_list:
    with open(files, 'r') as f:
        temp = json.load(f)
    obj_list.append(temp)
file_list = glob.glob(f'{file_path2}/*.json')
for files in file_list:
    with open(files, 'r') as f:
        temp = json.load(f)
    obj_list.append(temp)

Our first DataFrame: Emails

In [7]:

emails = pd.json_normalize(obj_list)
emails['record_type'] = 'email'
emails['record_type'] = emails['record_type'].astype('category')
emails

,filename,to,from,...,analysis.recruiter_name,analysis.recruiter_email,record_type
0,1926253.mime,naeem.paruk@outlook.com,Standard Bank <notification@smartrecruiters.com>,...,NaN,NaN,email
1,1926145.mime,naeem.paruk@outlook.com,Sinazo Dulana <candidate-42e0988c7654@archsoft...,...,NaN,NaN,email
2,56e7afb3-cfaf-493f-99fe-265a71fd2444.mime,NAEEM.PARUK@OUTLOOK.COM,info@jobalerts.pnet.co.za,...,NaN,NaN,email
3,1931200.mime,naeem.paruk@outlook.com,Psybergate <noreply@placementpartner.com>,...,NaN,NaN,email
4,1973844.mime,naeem.paruk@outlook.com,no-reply@agoda.global,...,NaN,NaN,email
...,...,...,...,...,...,...,...
3266,2213206.mime,naeem.paruk@outlook.com,DHL Express <NoReply.ODD@dhl.com>,...,NaN,NaN,email
3267,1932960.mime,naeem.paruk@outlook.com,The LEGO Group <lego@myworkday.com>,...,NaN,NaN,email
3268,2078481.mime,Naeem Paruk <naeem.paruk@outlook.com>,Microsoft Outlook Calendar <no-reply@microsoft...,...,NaN,NaN,email
3269,1936513.mime,naeem.paruk@outlook.com,Plum <no-reply@plum.io>,...,NaN,NaN,email


In [8]:
#Some further df cleaning

emails.columns = ['filename', # string
                'to', # string, email
                'from', # string, email 
                'subject', # string
                'cc', # string(s)
                'date', # ISO 8601 date
                'body', # string
                'company', # string
                'portal', # string (category)
                'notification_type', #string (category)
                'role', # string
                'update_category',# string (category)
                'is_automated', # boolean
                'urgency', # string (category)
                'interview_type', # string (category)
                'interview_platform', # string (category)
                'test_platform', # string (category)
                'test_duration_mins', # int
                'action_required', # boolean
                'action_type', # string (category)
                'summary', #string
                'interview_date', #ISO 8601 date
                'application_deadline', #ISO 8601 date
                'action_deadline',
                'work_mode', # string (category)
                'location', # string (category)
                'recruiter_name', # string
                'recruiter_email', # string
                'record_type'# string (category)
                ]
col_types = {'filename': str,
                'to': str,
                'from':'string',
                'subject':'string',
                'cc': str,
                'date': str,
                'body': str,
                'company': str,
                'portal': 'category',
                'notification_type':'category',
                'role':'string',
                'update_category':'category',
                'is_automated':bool,
                'urgency':'category',
                'interview_type':'category',
                'interview_platform':'category',
                'test_platform':'category',
                'test_duration_mins':int,
                'action_required':bool,
                'action_type':'category',
                'summary':'string',
                'interview_date':'string',
                'application_deadline':'string',
                'action_deadline':'string',
                'work_mode':'category',
                'location':'category',
                'recruiter_name':'string',
                'recruiter_email' :'string',
                'record_type':'category'
                }
timecols = ['date','interview_date','application_deadline','action_deadline']
emails = emails.astype(col_types, errors='ignore')

In [9]:
# Cleaning the dates
emails[timecols] = emails[timecols].apply(
    pd.to_datetime, format='mixed', errors='coerce', utc=True, cache=True
)

In [10]:
#drop the non-work related emails:
emails = emails[~emails['notification_type'].isin(['Unrelated', 'Job ad'])]
EMPTY_LIST = ['Unknown', 'None', None, 'nan', 'none', 'NA', 'NaN', ""]
#emails = emails[~emails['body'].isin(EMPTY_LIST)]
all_fields_empty = (
    emails['notification_type'].isin(['Unknown']) & 
    emails['company'].isin(EMPTY_LIST) & 
    emails['role'].isin(EMPTY_LIST) &
    ~emails['is_automated'] & 
    emails['portal'].isin(EMPTY_LIST) &
    emails['action_type'].isin(EMPTY_LIST)
)
emails = emails[~all_fields_empty]

emails.reset_index(drop=True, inplace=True)
emails

,filename,to,from,...,recruiter_name,recruiter_email,record_type
0,1926253.mime,naeem.paruk@outlook.com,Standard Bank <notification@smartrecruiters.com>,...,<NA>,<NA>,email
1,1926145.mime,naeem.paruk@outlook.com,Sinazo Dulana <candidate-42e0988c7654@archsoft...,...,<NA>,<NA>,email
2,1931200.mime,naeem.paruk@outlook.com,Psybergate <noreply@placementpartner.com>,...,<NA>,<NA>,email
3,1973844.mime,naeem.paruk@outlook.com,no-reply@agoda.global,...,<NA>,<NA>,email
4,1929937.mime,Naeem Paruk <naeem.paruk@outlook.com>,Signal Hill Products <notifications@app.bamboo...,...,<NA>,<NA>,email
...,...,...,...,...,...,...,...
924,1934443.mime,naeem.paruk@outlook.com,Communicate Recruitment <noreply@placementpart...,...,<NA>,<NA>,email
925,1931212.mime,naeem.paruk@outlook.com,Psybergate <noreply@placementpartner.com>,...,<NA>,<NA>,email
926,2184506.mime,naeem.paruk@outlook.com,Careers24 <no-reply@careers24.com>,...,<NA>,<NA>,email
927,1942604.mime,Naeem Paruk <naeem.paruk@outlook.com>,Alstom Recruiting Team <system@successfactors.eu>,...,<NA>,<NA>,email


In [11]:
# LinkedIn Quick Apply data
lnkdin = pd.read_csv('Linkedin_QuickApply.csv', index_col=0)

# Preserve employer screening questions before dropping feature columns
questions = lnkdin[[col for col in lnkdin.columns
                     if col.startswith('Ask_') or col.startswith('Req_')]].copy()
questions['job_url'] = lnkdin['Job_Url']

lnkdin.drop([col for col in lnkdin.columns
             if col.startswith('Ask_') or col.startswith('Req_')], axis=1, inplace=True)
lnkdin.columns = ['linkedin_date', 'company', 'role', 'job_url']

# LinkedIn exports dates in MM/DD/YYYY — dayfirst=False is correct.
# Using dayfirst=True previously swapped month/day for any day<=12,
# pushing April dates into future months (Oct, Aug, Nov, etc.).
lnkdin['linkedin_date'] = pd.to_datetime(lnkdin['linkedin_date'], dayfirst=False)
lnkdin

,linkedin_date,company,role,job_url
0,2026-04-17 12:17:00,Talent Glider,Supply Chain Analyst,http://www.linkedin.com/jobs/view/4400000696
1,2026-04-21 05:29:00,Pele Energy Group,Project Co-ordinator,http://www.linkedin.com/jobs/view/4404561257
2,2026-04-17 08:48:00,Optasia,"Business Intelligence Analyst, Fintech",http://www.linkedin.com/jobs/view/4402972316
3,2026-04-20 08:10:00,Clicks Group,Data Scientist (2 positions),http://www.linkedin.com/jobs/view/4400199144
4,2026-04-20 08:53:00,Metamorph,Junior Data Analyst,http://www.linkedin.com/jobs/view/4401601339
...,...,...,...,...
281,2026-04-03 04:11:00,Mackenzie Stuart,Graduate Business Consultant,http://www.linkedin.com/jobs/view/4396448827
282,2026-04-16 07:08:00,Conova Financial Recruitment,Global Equity Analyst,http://www.linkedin.com/jobs/view/4400408367
283,2026-03-30 00:40:00,Autochek Financial Services,Credit Analyst,http://www.linkedin.com/jobs/view/4395003734
284,2026-04-13 01:03:00,Allan Gray,Business Intelligence Developer,http://www.linkedin.com/jobs/view/4399511006


In [12]:
# Process emails to get jobs
emails.drop(['action_deadline','cc','recruiter_email', 'recruiter_name', 'location', 'work_mode', 'application_deadline', 'interview_date'], axis=1, inplace=True)

emails['uid'] = emails['filename'].str.split('.', expand=True)[0]
emails.drop(['filename', 'to', 'subject', 'body', 'test_platform', 'test_duration_mins', 'record_type'], inplace=True, axis=1)#Remove this at some point

In [13]:
emails = emails.reindex(columns=['uid', 'date', 'notification_type', 'update_category', 'company', 'role', 'urgency', 'action_required', 'action_type', 'is_automated', 'portal', 'from',  'interview_platform', 'interview_type', 'summary' ])
emails.sort_values('company', inplace=True)
emails.reset_index(drop=True, inplace=True)
emails['company']=emails['company'].replace("Unknown",None)


In [ ]:
df = parse_jobs('nonfunctionapps/linkedindump.txt')
df.sort_values('company', inplace=True)
df.reset_index(drop=True, inplace=True)
emails.sort_values('company', inplace=True)
emails.reset_index(drop=True, inplace=True)
lnkdin.sort_values('company', inplace=True)
lnkdin.reset_index(drop=True, inplace=True)
df.to_csv('df.csv')
lnkdin.to_csv('lnkdin.csv')
emails.to_csv('emails.csv')



FileNotFoundError: [Errno 2] No such file or directory: 'linkedindump.txt'